# Time since fire at ignition, 2015–2026

For each fire year **Y**:

1. Keep only fires that burned in year Y (MODIS pixels + MTBS named perimeters).
2. Assign an **exact date** from MODIS `BurnDate` (day of year) or MTBS `Ig_Date`.
3. Re-run TSF for those perimeters as of **31 Dec (Y−1)** using the full MODIS history (2000 through that December). Example: 2020 fires use 2019-12-31.
4. Add `(fire_date − 31 Dec Y−1)` to get **TSF at ignition**.

Put `export_tsf_by_year.py` and `export_time_since_fire.py` next to this notebook. Earth Engine project: `timesincefire`.

Outputs go to `new_data/Time_since_fire/by_year_2015_2026/`.

If 2015–2020 already ran, set `START_YEAR = 2021` so only the new years are built; the copy cell merges them. On the website: **Fires**, then pick a year through 2026.

In [15]:
from pathlib import Path
import sys

# 2015-2020 already built? keep 2021 so this run only adds the new years.
START_YEAR = 2015
END_YEAR = 2026

here = Path.cwd().resolve()
sys.path.insert(0, str(here))

def find_tsf_dir():
    checks = []
    for parent in [here, *here.parents]:
        checks.append(parent / 'new_data' / 'Time_since_fire')
        checks.append(parent / 'Time_since_fire')
    checks.extend([
        Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire'),
        Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire'),
        Path('/home/jovyan/data-store/new_data/Time_since_fire'),
    ])
    for p in checks:
        if p.exists():
            return p
    return Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire')

TSF = find_tsf_dir()
OUT = TSF / 'by_year_2015_2026'
print('TSF folder:', TSF, 'exists' if TSF.exists() else 'will be created')
print('Output:', OUT)

TSF folder: /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire exists
Output: /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire/by_year_2015_2026


In [16]:
import shutil, subprocess, sys
from pathlib import Path

def have_ee():
    try:
        import ee  # noqa: F401
        return True
    except ModuleNotFoundError:
        return False

def run(cmd):
    print('+', ' '.join(str(c) for c in cmd))
    return subprocess.call(cmd)

if have_ee():
    import ee
    print('earthengine-api OK', getattr(ee, '__version__', ''))
else:
    conda = shutil.which('conda') or '/opt/conda/bin/conda'
    pip = shutil.which('pip') or shutil.which('pip3') or '/opt/conda/bin/pip'
    ok = 1
    if Path(conda).exists():
        ok = run([conda, 'install', '-y', '-c', 'conda-forge', 'earthengine-api', 'geemap'])
    if ok != 0 and Path(pip).exists():
        ok = run([pip, 'install', 'earthengine-api', 'geemap'])
    if ok != 0:
        run([sys.executable, '-m', 'ensurepip', '--upgrade'])
        ok = run([sys.executable, '-m', 'pip', 'install', 'earthengine-api', 'geemap'])
    if not have_ee():
        raise ModuleNotFoundError(
            'Packages installed, but this kernel cannot see them yet. '
            'Kernel → Restart Kernel (this notebook only), then re-run this cell. '
            'Do not Shut Down All Kernels — that would stop MESMA in the other terminal.'
        )
    import ee
    print('earthengine-api OK', getattr(ee, '__version__', ''))

earthengine-api OK 1.7.53


In [17]:
from IPython import get_ipython

try:
    import ee  # noqa: F401
except ModuleNotFoundError as e:
    raise ModuleNotFoundError('earthengine-api is not installed. Re-run the cell above.') from e

script = here / 'export_tsf_by_year.py'
if not script.exists():
    script = Path('/home/jovyan/data-store/sce_site/export_tsf_by_year.py')
if not script.exists():
    raise FileNotFoundError('Upload export_tsf_by_year.py next to this notebook')

cmd = (
    f'{script} --start-year {START_YEAR} --end-year {END_YEAR} --out {OUT}'
)
print('Running', cmd)
get_ipython().run_line_magic('run', cmd)

Running /home/jovyan/data-store/sce_site/export_tsf_by_year.py --start-year 2015 --end-year 2026 --out /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire/by_year_2015_2026
Earth Engine project: timesincefire
Years 2015–2026  →  /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire/by_year_2015_2026

=== 2015  TSF as of 2014-12-31 ===
  exporting 2015_modis_burn_date.tif ...
Generating URL ...
Please wait ...
Data downloaded to /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire/by_year_2015_2026/2015_modis_burn_date.tif
  exporting 2015_tsf_as_of_dec_prev.tif ...
Generating URL ...
Please wait ...
Data downloaded to /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire/by_year_2015_2026/2015_tsf_as_of_dec_prev.tif
  exporting 2015_days_from_dec_to_fire.tif ...
Generating URL ...
Please wait ...
Data downloaded to /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/n

In [18]:
# Finish fires.geojson after a data-store I/O error. Do not re-run Earth Engine.
import json, os, time, shutil
from pathlib import Path

KEEP = (
    'burn_year', 'burn_date', 'Incid_Name', 'tsf_at_ignition',
    'tsf_as_of_dec_prev', 'days_from_dec_prev', 'as_of_date',
    'BurnBndAc', 'date_source',
)
LOCAL = Path('/tmp/tsf_merge')
LOCAL.mkdir(exist_ok=True)

def wait_listdir(folder, tries=15):
    for i in range(tries):
        try:
            names = os.listdir(folder)
            print(f'listed {folder}: {len(names)} files')
            return names
        except OSError as exc:
            print(f'wait {i+1}/{tries}: {exc}')
            time.sleep(10)
    print('directory still wedged; will try known filenames only')
    return []

def read_text(path, tries=6):
    for i in range(tries):
        try:
            return path.read_text()
        except FileNotFoundError:
            print('skip missing', path.name)
            return None
        except OSError as exc:
            if getattr(exc, 'errno', None) == 2:
                print('skip missing', path.name)
                return None
            print(f'read {path.name} {i+1}/{tries}: {exc}')
            time.sleep(5)
    return None

wait_listdir(OUT)

candidates = [
    OUT / f'fires_{START_YEAR}_{END_YEAR}.geojson',
    TSF / 'by_year_2015_2020' / 'fires_2015_2020.geojson',
    TSF / 'by_year_2015_2020' / 'fires_web.geojson',
    TSF / 'by_year_2015_2020' / 'fires.geojson',
]
for year in range(2015, 2027):
    candidates.append(OUT / f'{year}_mtbs_perimeters.geojson')
    candidates.append(OUT / f'{year}_modis_perimeters.geojson')

slim, seen = [], set()
for src in candidates:
    text = read_text(src)
    if not text:
        continue
    try:
        feats = json.loads(text).get('features') or []
    except json.JSONDecodeError as exc:
        print('bad json', src.name, exc)
        continue
    print(f'merge {src.name}: {len(feats)}')
    for feat in feats:
        props = feat.get('properties') or {}
        geom = feat.get('geometry')
        if not geom:
            continue
        key = (props.get('Incid_Name'), props.get('burn_year'), props.get('burn_date'), props.get('date_source'))
        if key in seen:
            continue
        seen.add(key)
        slim.append({'type': 'Feature', 'properties': {k: props[k] for k in KEEP if k in props}, 'geometry': geom})

payload = json.dumps({'type': 'FeatureCollection', 'features': slim}, separators=(',', ':'))
local = LOCAL / 'fires.geojson'
local.write_text(payload)
print('local copy', local, len(slim), 'fires')

def copy_out(dest):
    for i in range(8):
        try:
            shutil.copyfile(local, dest)
            print('wrote', dest)
            return
        except OSError as exc:
            print(f'write {dest.name} {i+1}/8: {exc}')
            time.sleep(10)
    print('could not write', dest, '— use the local copy at', local)

copy_out(OUT / 'fires.geojson')
copy_out(OUT / 'fires_web.geojson')

listed /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/Time_since_fire/by_year_2015_2026: 80 files
merge fires_2015_2026.geojson: 1448
skip missing fires_2015_2020.geojson
skip missing fires_web.geojson
skip missing fires.geojson
merge 2015_mtbs_perimeters.geojson: 18
merge 2015_modis_perimeters.geojson: 62
merge 2016_mtbs_perimeters.geojson: 33
merge 2016_modis_perimeters.geojson: 94
merge 2017_mtbs_perimeters.geojson: 37
merge 2017_modis_perimeters.geojson: 137
merge 2018_mtbs_perimeters.geojson: 19
merge 2018_modis_perimeters.geojson: 84
merge 2019_mtbs_perimeters.geojson: 14
merge 2019_modis_perimeters.geojson: 127
merge 2020_mtbs_perimeters.geojson: 45
merge 2020_modis_perimeters.geojson: 153
merge 2021_mtbs_perimeters.geojson: 17
merge 2021_modis_perimeters.geojson: 95
merge 2022_mtbs_perimeters.geojson: 6
merge 2022_modis_perimeters.geojson: 35
merge 2023_mtbs_perimeters.geojson: 15
merge 2023_modis_perimeters.geojson: 113
merge 2024_mtbs_perimeters.geojson: 3


In [19]:
import shutil
src = OUT / f'fires_{START_YEAR}_{END_YEAR}.geojson'
shutil.copyfile(src, OUT / 'fires.geojson')
shutil.copyfile(src, OUT / 'fires_web.geojson')
print('copied', src.name, '→ fires.geojson')

copied fires_2015_2026.geojson → fires.geojson


In [20]:
from pathlib import Path

print('Wrote:')
if OUT.exists():
    for p in sorted(OUT.iterdir()):
        if p.is_file():
            print(f'  {p.name:42s} {p.stat().st_size/1e6:6.2f} MB')
else:
    print('  (no output folder yet)')

combo = OUT / 'fires.geojson'
if not combo.exists():
    combo = OUT / f'fires_{START_YEAR}_{END_YEAR}.geojson'
if combo.exists():
    import json
    feats = json.loads(combo.read_text()).get('features', [])
    print(f'\nMTBS named fires: {len(feats)}')
    for f in feats[:8]:
        p = f.get('properties', {})
        print(
            f"  {p.get('burn_year')}  {p.get('Incid_Name', '?'):20s}  "
            f"{p.get('burn_date')}  TSF@ignition={p.get('tsf_at_ignition')}"
        )

Wrote:
  2015_days_from_dec_to_fire.tif               0.02 MB
  2015_modis_burn_date.tif                     0.02 MB
  2015_modis_perimeters.geojson                0.06 MB
  2015_mtbs_perimeters.geojson                 0.75 MB
  2015_tsf_as_of_dec_prev.tif                  0.01 MB
  2015_tsf_at_ignition.tif                     0.01 MB
  2016_days_from_dec_to_fire.tif               0.02 MB
  2016_modis_burn_date.tif                     0.02 MB
  2016_modis_perimeters.geojson                0.11 MB
  2016_mtbs_perimeters.geojson                 1.24 MB
  2016_tsf_as_of_dec_prev.tif                  0.02 MB
  2016_tsf_at_ignition.tif                     0.02 MB
  2017_days_from_dec_to_fire.tif               0.02 MB
  2017_modis_burn_date.tif                     0.02 MB
  2017_modis_perimeters.geojson                0.13 MB
  2017_mtbs_perimeters.geojson                 1.71 MB
  2017_tsf_as_of_dec_prev.tif                  0.02 MB
  2017_tsf_at_ignition.tif                     0.02 MB
  2

## Copy onto the website

Writes `atlas/tsf_years/fires.geojson` (2015–2026, merged with any 2015–2020 run) and copies the scene HTML. Then open a tile and click **Fires**.

In [21]:
import json
import shutil
from pathlib import Path

html_src = here / 'postfire_landscape_atlas1.html'
if not html_src.exists():
    html_src = Path('/home/jovyan/data-store/sce_site/postfire_landscape_atlas1.html')

atlas = None
for cand in [
    TSF.parent / '2026-07-28' / 'atlas',
    Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/atlas'),
    Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/atlas'),
    Path('/home/jovyan/data-store/new_data/2026-07-28/atlas'),
]:
    if cand.exists():
        atlas = cand
        break
if atlas is None:
    raise FileNotFoundError('Could not find atlas/ — run the 2026-07-28 pipeline first')

src = OUT / 'fires.geojson'
if not src.exists():
    src = OUT / f'fires_{START_YEAR}_{END_YEAR}.geojson'
if not src.exists():
    raise FileNotFoundError('No year GeoJSON yet — run the export cell first')
dest = atlas / 'tsf_years' / 'fires.geojson'
dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(src, dest)
n = len(json.loads(src.read_text()).get('features') or [])
print(f'Wrote {n} fires to {dest}')

if html_src.exists():
    text = html_src.read_text()
    n = 0
    for d in sorted(p for p in atlas.iterdir() if p.is_dir()):
        if (d / 'mesma_grid.js').exists():
            (d / 'postfire_landscape_atlas1.html').write_text(text)
            n += 1
    print(f'Copied year-tab HTML into {n} tile folders')
else:
    print('Upload the new postfire_landscape_atlas1.html next to this notebook')

print('Open atlas/index.html, pick a scene, then Map layers → Fires (2015–2026).')

Wrote 1448 fires to /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/atlas/tsf_years/fires.geojson
Copied year-tab HTML into 41 tile folders
Open atlas/index.html, pick a scene, then Map layers → Fires (2015–2026).
